# CRISP-DM Telco Customer Churn AnalysisDataset: [Telco Customer Churn - Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)This notebook follows the **CRISP-DM** methodology end-to-end for a churn classification problem.

## Phase Map- **Business Understanding**- **Data Understanding**- **Data Preparation**- **Modeling**- **Evaluation**- **Deployment**

> Set the project context, business objectives, and success criteria before touching data.

## Business Understanding**Goal**: Reduce customer churn by predicting which customers are likely to leave the telecom service.**Key Questions**- What are the business drivers of churn?- What monetary impact would a 5% reduction in churn have?- Which customer segments are most at risk?**Success Criteria**- Primary: Recall ≥ 0.80 on hold-out data.- Secondary: Actionable segmentation insights for retention campaigns.

In [ ]:
project_charter = {
    'stakeholders': ['VP Customer Success', 'Retention Analytics Lead', 'Data Engineering'],
    'business_objectives': ['Quantify churn risk', 'Prioritize outreach campaigns'],
    'constraints': ['Data refresh monthly', 'Model must be explainable'],
    'milestones': {
        'kickoff': 'Define outcomes and KPIs',
        'baseline_model': 'First iteration with classical ML',
        'deployment_candidate': 'Pipeline validated in UAT'
    }
}
project_charter


## Data Understanding**Inputs**- Kaggle dataset `Telco-Customer-Churn.csv` (download via Kaggle API)- Data dictionary from Kaggle page**Activities**1. Inspect schema, datatypes, and missingness.2. Explore churn rate and categorical distributions.3. Visualize relationships between tenure, contract type, and churn.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

raw_path = Path('../data/raw/Telco-Customer-Churn.csv')
if not raw_path.exists():
    raise FileNotFoundError('Download Telco-Customer-Churn.csv into data/raw before running.')

data = pd.read_csv(raw_path)
data.head()


In [ ]:
summary = data.describe(include='all').transpose()
summary[['count', 'unique', 'top', 'freq']].head(10)


In [ ]:
churn_rate = data['Churn'].value_counts(normalize=True)
print('Churn distribution:\n', churn_rate)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=data, x='Churn', ax=ax[0])
ax[0].set_title('Churn Distribution')

# Class imbalance visualization
churn_counts = data['Churn'].value_counts()
ax[1].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%', startangle=90, colors=['#2ecc71', '#e74c3c'])
ax[1].set_title('Churn Proportion')
plt.tight_layout()
plt.show()

print(f'\nClass Imbalance Ratio: {churn_counts["No"] / churn_counts["Yes"]:.2f}:1')

### Relationship PlotsThese visuals support the narrative in the Medium article. Add additional plots as needed.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

# Tenure vs Churn
sns.histplot(data=data, x='tenure', hue='Churn', multiple='stack', ax=ax[0, 0], bins=30)
ax[0, 0].set_title('Tenure vs Churn')
ax[0, 0].set_xlabel('Tenure (months)')

# Contract Type vs Churn
sns.countplot(data=data, x='Contract', hue='Churn', ax=ax[0, 1])
ax[0, 1].set_title('Contract Type vs Churn')
ax[0, 1].tick_params(axis='x', rotation=15)

# Monthly Charges vs Churn
sns.boxplot(data=data, x='Churn', y='MonthlyCharges', ax=ax[1, 0])
ax[1, 0].set_title('Monthly Charges Distribution by Churn')

# Internet Service vs Churn
sns.countplot(data=data, x='InternetService', hue='Churn', ax=ax[1, 1])
ax[1, 1].set_title('Internet Service vs Churn')
ax[1, 1].tick_params(axis='x', rotation=15)

fig.tight_layout()
plt.show()

# Churn rate by key categories
print('\n=== Churn Rates by Key Features ===')
for col in ['Contract', 'InternetService', 'PaymentMethod']:
    print(f'\n{col}:')
    churn_by_cat = data.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    print(churn_by_cat.sort_values(ascending=False))

### Correlation Analysis
Examine relationships between numeric features and churn to identify potential predictors.

In [ ]:
# Create numeric version of dataset for correlation analysis
data_numeric = data.copy()
data_numeric['Churn_Binary'] = (data['Churn'] == 'Yes').astype(int)

# Convert categorical to numeric where meaningful
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    if col in data_numeric.columns:
        data_numeric[f'{col}_num'] = (data_numeric[col] == 'Yes').astype(int)

# Select numeric columns for correlation
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_Binary'] + \
                   [f'{col}_num' for col in binary_cols if col in data_numeric.columns]

correlation_matrix = data_numeric[numeric_features].corr()

# Plot correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Numeric Features', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Top correlations with churn
print('\n=== Features Most Correlated with Churn ===')
churn_corr = correlation_matrix['Churn_Binary'].drop('Churn_Binary').abs().sort_values(ascending=False)
print(churn_corr.head(8))

## Data Preparation**Plan**- Convert `TotalCharges` to numeric.- Impute or drop missing tenure values.- Encode categorical features using One-Hot Encoding.- Standardize continuous features.- Split data into train/validation/test respecting class balance.

In [ ]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
missing_total = data['TotalCharges'].isna().sum()
print(f'TotalCharges missing: {missing_total}')
data['TotalCharges'] = data['TotalCharges'].fillna(data['TotalCharges'].median())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

target = 'Churn'
X = data.drop(columns=[target, 'customerID'])
y = data[target].map({'No': 0, 'Yes': 1})
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object']).columns.tolist()

preprocess = ColumnTransformer(transformers=[
    ('cat', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols),
    ('num', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols)
])

X_train, X_temp, y_train, y_temp = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, stratify=y_temp, test_size=0.5, random_state=42)
X_train.shape, X_valid.shape, X_test.shape


## ModelingStart with an interpretable baseline (Logistic Regression) and iterate with ensemble models. Track experiments in the critique logs.

### Advanced Ensemble Models
Compare with gradient boosting algorithms (XGBoost, LightGBM) for potential performance gains.

In [ ]:
# Install if needed: !pip install xgboost lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# XGBoost Model
xgb_model = Pipeline(steps=[
    ('prep', preprocess), 
    ('model', XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=3,  # Handle class imbalance
        random_state=42,
        eval_metric='logloss'
    ))
])
xgb_model.fit(X_train, y_train)
xgb_valid_preds = xgb_model.predict(X_valid)
xgb_valid_proba = xgb_model.predict_proba(X_valid)[:, 1]
print('=== XGBoost Performance ===')
print(classification_report(y_valid, xgb_valid_preds))
print('Validation ROC-AUC:', roc_auc_score(y_valid, xgb_valid_proba))

# LightGBM Model
lgbm_model = Pipeline(steps=[
    ('prep', preprocess),
    ('model', LGBMClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    ))
])
lgbm_model.fit(X_train, y_train)
lgbm_valid_preds = lgbm_model.predict(X_valid)
lgbm_valid_proba = lgbm_model.predict_proba(X_valid)[:, 1]
print('\n=== LightGBM Performance ===')
print(classification_report(y_valid, lgbm_valid_preds))
print('Validation ROC-AUC:', roc_auc_score(y_valid, lgbm_valid_proba))

### Model Interpretability with SHAP
Use SHAP values to explain model predictions and identify key drivers of churn.

In [ ]:
from joblib import dump
import json
from datetime import datetime

artifacts_dir = Path('../app/artifacts')
artifacts_dir.mkdir(exist_ok=True, parents=True)

# Save the champion model (update based on comparison results)
# For this example, we'll save the best performing model
best_model = rf_model  # Replace with champion from comparison
model_path = artifacts_dir / 'telco_churn_pipeline.joblib'
dump(best_model, model_path)
print(f'✓ Pipeline saved to {model_path}')

# Save feature names for reference
feature_metadata = {
    'categorical_features': categorical_cols,
    'numeric_features': numeric_cols,
    'target': target
}
with open(artifacts_dir / 'feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)
print(f'✓ Feature metadata saved')

# Create Model Card
model_card = {
    'model_name': 'Telco Customer Churn Predictor',
    'version': '1.0.0',
    'created_date': datetime.now().isoformat(),
    'methodology': 'CRISP-DM',
    'algorithm': 'Random Forest Classifier',  # Update based on champion
    'training_data': {
        'source': 'Kaggle - Telco Customer Churn',
        'n_samples_train': len(X_train),
        'n_samples_valid': len(X_valid),
        'n_samples_test': len(X_test),
        'class_distribution': f'{(y_train == 0).sum()} No Churn, {(y_train == 1).sum()} Churn'
    },
    'performance_metrics': {
        'test_accuracy': accuracy_score(y_test, test_preds),
        'test_precision': precision_score(y_test, test_preds),
        'test_recall': recall_score(y_test, test_preds),
        'test_f1': f1_score(y_test, test_preds),
        'test_roc_auc': roc_auc_score(y_test, test_proba)
    },
    'business_requirements': {
        'primary_metric': 'Recall >= 0.80',
        'objective': 'Identify at-risk customers for retention campaigns',
        'inference_latency_target': '<150ms per prediction'
    },
    'limitations': [
        'Model trained on single telecom dataset - may not generalize to other industries',
        'Class imbalance may affect precision',
        'Requires monthly retraining as customer behavior changes',
        'Does not account for seasonality or external economic factors'
    ],
    'ethical_considerations': [
        'Ensure fair treatment across demographic groups',
        'Avoid discriminatory targeting based on protected attributes',
        'Provide transparency to customers about retention offers'
    ],
    'usage': {
        'input_format': 'JSON with customer attributes',
        'output_format': 'Churn probability (0-1) and risk category',
        'deployment_endpoint': '/score'
    }
}

with open(artifacts_dir / 'model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2)
print(f'✓ Model card saved to {artifacts_dir / "model_card.json"}')

print(f'\n📦 Deployment artifacts ready in {artifacts_dir}/')

### Business Impact & Cost-Benefit Analysis
Translate model performance into business value with retention economics.

In [ ]:
# Business parameters (hypothetical but realistic)
avg_customer_lifetime_value = 1200  # Average LTV per customer
retention_campaign_cost = 100  # Cost per targeted customer
retention_success_rate = 0.35  # 35% of contacted at-risk customers are retained

# Get confusion matrix values from test set
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()

print('=== Confusion Matrix Breakdown ===')
print(f'True Negatives (Correctly predicted no churn): {tn}')
print(f'False Positives (Predicted churn, actually stayed): {fp}')
print(f'False Negatives (Predicted no churn, actually churned): {fn}')
print(f'True Positives (Correctly predicted churn): {tp}')

# Calculate business metrics
customers_targeted = tp + fp  # All predicted churners
wasted_campaigns = fp  # False alarms
missed_opportunities = fn  # Churners we didn't catch

# Revenue saved (customers we correctly identified and retained)
customers_saved = tp * retention_success_rate
revenue_saved = customers_saved * avg_customer_lifetime_value

# Costs
campaign_costs = customers_targeted * retention_campaign_cost

# Revenue lost (customers who churned despite intervention + missed customers)
customers_lost = fn + (tp * (1 - retention_success_rate))
revenue_lost = customers_lost * avg_customer_lifetime_value

# Net benefit
net_benefit = revenue_saved - campaign_costs

print(f'\n=== Business Impact Analysis ===')
print(f'Customers Targeted for Retention: {customers_targeted}')
print(f'Estimated Customers Saved: {customers_saved:.1f}')
print(f'Revenue Saved: ${revenue_saved:,.2f}')
print(f'Campaign Costs: ${campaign_costs:,.2f}')
print(f'Revenue Lost (Missed + Failed Retention): ${revenue_lost:,.2f}')
print(f'\n💰 Net Benefit: ${net_benefit:,.2f}')
print(f'📊 ROI: {(net_benefit / campaign_costs * 100):.1f}%')

# Visualize business impact
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Cost-Benefit breakdown
categories = ['Revenue\nSaved', 'Campaign\nCosts', 'Net\nBenefit']
values = [revenue_saved, -campaign_costs, net_benefit]
colors = ['#2ecc71', '#e74c3c', '#3498db']

bars = ax[0].bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
ax[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax[0].set_ylabel('Amount ($)')
ax[0].set_title('Cost-Benefit Analysis', fontsize=12, fontweight='bold')
ax[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax[0].text(bar.get_x() + bar.get_width()/2., height,
               f'${abs(height):,.0f}',
               ha='center', va='bottom' if height > 0 else 'top', fontweight='bold')

# Customer flow Sankey-style visualization
customer_categories = ['True\nPositives', 'False\nPositives', 'False\nNegatives', 'True\nNegatives']
customer_counts = [tp, fp, fn, tn]
category_colors = ['#2ecc71', '#f39c12', '#e74c3c', '#95a5a6']

bars2 = ax[1].barh(customer_categories, customer_counts, color=category_colors, alpha=0.7, edgecolor='black')
ax[1].set_xlabel('Number of Customers')
ax[1].set_title('Customer Classification Breakdown', fontsize=12, fontweight='bold')
ax[1].grid(axis='x', alpha=0.3)

# Add count labels
for i, (bar, count) in enumerate(zip(bars2, customer_counts)):
    ax[1].text(count + 10, i, f'{count}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Model Comparison
Systematically compare all models across key metrics to select the champion.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Collect all model predictions
models_dict = {
    'Logistic Regression': (valid_preds, valid_proba),
    'Random Forest': (rf_valid_preds, rf_valid_proba),
    'XGBoost': (xgb_valid_preds, xgb_valid_proba),
    'LightGBM': (lgbm_valid_preds, lgbm_valid_proba)
}

# Calculate metrics for each model
comparison_results = []
for name, (preds, proba) in models_dict.items():
    comparison_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_valid, preds),
        'Precision': precision_score(y_valid, preds),
        'Recall': recall_score(y_valid, preds),
        'F1-Score': f1_score(y_valid, preds),
        'ROC-AUC': roc_auc_score(y_valid, proba)
    })

comparison_df = pd.DataFrame(comparison_results)
print('=== Model Performance Comparison ===')
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Metric comparison bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
comparison_df.set_index('Model')[metrics_to_plot].plot(kind='bar', ax=ax[0], width=0.8)
ax[0].set_title('Model Performance Metrics Comparison', fontsize=12, fontweight='bold')
ax[0].set_ylabel('Score')
ax[0].set_xlabel('Model')
ax[0].legend(loc='lower right')
ax[0].set_ylim(0, 1)
ax[0].grid(axis='y', alpha=0.3)
ax[0].tick_params(axis='x', rotation=45)

# ROC curves comparison
for name, (_, proba) in models_dict.items():
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_valid, proba)
    ax[1].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_valid, proba):.3f})')

ax[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC Curves Comparison', fontsize=12, fontweight='bold')
ax[1].legend(loc='lower right')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Select champion model (highest recall as per business requirement)
champion_model = comparison_df.loc[comparison_df['Recall'].idxmax(), 'Model']
print(f'\n🏆 Champion Model: {champion_model} (Highest Recall: {comparison_df["Recall"].max():.3f})')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

log_reg = Pipeline(steps=[('prep', preprocess), ('model', LogisticRegression(max_iter=1000))])
log_reg.fit(X_train, y_train)
valid_preds = log_reg.predict(X_valid)
valid_proba = log_reg.predict_proba(X_valid)[:, 1]
print(classification_report(y_valid, valid_preds))
print('Validation ROC-AUC:', roc_auc_score(y_valid, valid_proba))


In [ ]:
rf_model = Pipeline(steps=[('prep', preprocess), ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))])
rf_model.fit(X_train, y_train)
rf_valid_preds = rf_model.predict(X_valid)
rf_valid_proba = rf_model.predict_proba(X_valid)[:, 1]
print(classification_report(y_valid, rf_valid_preds))
print('Validation ROC-AUC:', roc_auc_score(y_valid, rf_valid_proba))


## EvaluationEvaluate on the untouched test set and capture key charts (ROC, precision-recall, calibration).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

test_proba = rf_model.predict_proba(X_test)[:, 1]
test_preds = (test_proba >= 0.5).astype(int)
print(classification_report(y_test, test_preds))
print('Test ROC-AUC:', roc_auc_score(y_test, test_proba))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_preds, ax=ax[0])
ax[0].set_title('Confusion Matrix')
RocCurveDisplay.from_predictions(y_test, test_proba, ax=ax[1])
ax[1].set_title('ROC Curve')
PrecisionRecallDisplay.from_predictions(y_test, test_proba, ax=ax[2])
ax[2].set_title('Precision-Recall Curve')
fig.tight_layout()
plt.show()


## DeploymentPackage the trained pipeline, document assumptions, and expose a scoring endpoint via FastAPI (see `/app`).

In [ ]:
from joblib import dump
artifacts_dir = Path('../app/artifacts')
artifacts_dir.mkdir(exist_ok=True, parents=True)
model_path = artifacts_dir / 'telco_churn_pipeline.joblib'
dump(rf_model, model_path)
print(f'Pipeline saved to {model_path}')


In [ ]:
sample_payload = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 12,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 80.65,
    'TotalCharges': 1020.5
}
sample_payload


### Next Steps- Compare with gradient boosted trees (e.g., XGBoost, LightGBM).- Incorporate cost-sensitive threshold tuning.- Back-test retention strategies with marketing team.Update the prompts in `prompts/` after each phase is reviewed by GPT-5/Claude.